# Train ranker v1 (LightGBM vs formula baseline)

ТЗ-2: обучение бинарного LightGBM на `ml/data/synthetic_v1.csv`, сравнение с `formula_score`,
калибровка, SHAP, oracle-анализ латенток. Артефакты пишутся **только если все гейты зелёные**.

Запуск из `ml/` (uv, не pip):

```bash
cd ml
uv sync
uv run python -m nbconvert --execute --to notebook --inplace \
  --ExecutePreprocessor.timeout=600 \
  notebooks/train_ranker.ipynb
# или без Jupyter:
uv run python notebooks/ranker_train.py
```

Seed везде `42`. Тюнинга на test нет.

## 1. Контракты и загрузка

Читаем `ml/features.json` и CSV. Assert: колонки после `raw_json` совпадают с манифестом
по именам и порядку. Латентки (`r`, `m`, `c_e`, `fraud`, `epsilon`, `pop`) не должны
быть колонками фич. `y = (label == COMPLETED)`, `X` — 16 колонок манифеста.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
nb_dir = cwd / "ml" / "notebooks"
if not nb_dir.exists():
    nb_dir = cwd  # launched from ml/notebooks
sys.path.insert(0, str(nb_dir))

import ranker_train as rt

rt.set_seeds(rt.SEED)
ROOT = rt.find_root()
manifest, df = rt.load_contracts(ROOT)
X, y, formula = rt.make_xy(df, manifest)
print("root:", ROOT)
print("rows:", len(df), "features:", len(manifest))
print("P(COMPLETED) =", float(y.mean()))
print("manifest:", manifest)
assert list(X.columns) == manifest
assert "formula_score" in df.columns

root: /home/dan/projects/tricky-exchanger
rows: 20708 features: 16
P(COMPLETED) = 0.3468224840641298
manifest: ['match_mean', 'min_edge', 'edge_spread', 'liquidity_min', 'liquidity_mean', 'size_spread', 'count', 'progress', 'is_proposed', 'is_frozen', 'hours_since_created', 'hours_in_stage', 'vote_velocity', 'category_popularity', 'category_diversity', 'reliability_mean']


/home/dan/projects/tricky-exchanger/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Split

Train/test 80/20, стратификация по `(y, stage)`. Из train ещё 80/20 на fit/val
для early stopping. Test в тюнинге не участвует.

In [2]:
split = rt.stratified_split(df, X, y, formula, rt.SEED)
print("fit", len(split["fit"]), "val", len(split["val"]), "test", len(split["test"]))
print("test stages:\n", split["df_test"]["stage"].value_counts().to_string())

fit 13252 val 3314 test 4142
test stages:
 stage
CANDIDATE    2917
PROPOSED      727
FROZEN        498


## 3. Обучение LightGBM

`objective=binary`, метрики `auc` + `binary_logloss`, early stopping по val log-loss
(patience 50). Умеренные гиперпараметры: `n_estimators≤500`, `lr=0.03`,
`num_leaves=16`, `min_data_in_leaf=80`, `reg_lambda=1.0`, `seed=42`
(чуть сильнее регуляризация, чтобы калибровка ±5% держалась после стадии IN_PROGRESS).
Тюнинга на test нет.

In [3]:
model = rt.train_lgbm(split, rt.SEED)
print("best_iteration", getattr(model, "best_iteration_", None))
p_test = model.predict_proba(split["X_test"])[:, 1]
p_base = split["formula_test"]

best_iteration 500


## 4. Сравнение с baseline на test

AUC и log-loss: LGBM vs `formula_score`. Срезы: ADD (ранжирование кандидатов),
PROPOSED, FROZEN, IN_PROGRESS.

In [4]:
metrics = rt.slice_metrics(split["df_test"], split["y_test"], p_test, p_base)
print(rt.metrics_markdown(metrics))

| slice | n | AUC LGBM | AUC formula | Δ AUC | logloss LGBM | logloss formula |
|---|---:|---:|---:|---:|---:|---:|
| overall | 4142 | 0.8801 | 0.7501 | +0.1300 | 0.4079 | 0.5991 |
| ADD | 2007 | 0.8109 | 0.6167 | +0.1941 | 0.4020 | 0.5760 |
| PROPOSED | 727 | 0.8511 | 0.5609 | +0.2902 | 0.4997 | 0.6859 |
| FROZEN | 498 | 0.8361 | 0.4721 | +0.3640 | 0.3716 | 0.4904 |


## 5. Калибровка

Reliability diagram, 10 бинов. Гейт: |predicted − empirical| ≤ 5% в каждом бине
с поддержкой ≥ 30 строк.

In [5]:
fig_dir = ROOT / "ml" / "reports" / "figures"
calib = rt.reliability_table(split["y_test"], p_test)
rt.plot_reliability(calib, fig_dir / "reliability.png")
print(rt.df_to_markdown(calib))

| bin | lo | hi | n | predicted | empirical | abs_err | supported |
|---|---|---|---|---|---|---|---|
| 0 | 0.0000 | 0.1000 | 1096 | 0.0480 | 0.0310 | 0.0170 | True |
| 1 | 0.1000 | 0.2000 | 633 | 0.1472 | 0.1280 | 0.0192 | True |
| 2 | 0.2000 | 0.3000 | 516 | 0.2472 | 0.2306 | 0.0166 | True |
| 3 | 0.3000 | 0.4000 | 389 | 0.3476 | 0.3573 | 0.0097 | True |
| 4 | 0.4000 | 0.5000 | 293 | 0.4491 | 0.4573 | 0.0082 | True |
| 5 | 0.5000 | 0.6000 | 264 | 0.5481 | 0.5530 | 0.0049 | True |
| 6 | 0.6000 | 0.7000 | 220 | 0.6518 | 0.6227 | 0.0290 | True |
| 7 | 0.7000 | 0.8000 | 247 | 0.7501 | 0.7935 | 0.0435 | True |
| 8 | 0.8000 | 0.9000 | 299 | 0.8512 | 0.8963 | 0.0451 | True |
| 9 | 0.9000 | 1.0000 | 185 | 0.9349 | 0.9838 | 0.0489 | True |


## 6. SHAP

Beeswarm + dependence для топ-4. Ожидаемые эффекты DGP: порог `min_edge`,
пенальти `count=4`, насыщение `liquidity_min` на 1.0, взаимодействие
`match_mean × liquidity`. `reliability_mean` — константа 0.75, важность ≈ 0.

In [6]:
shap_info = rt.shap_bundle(model, split["X_test"], fig_dir)
print("top4:", shap_info["top4"])
print("reliability_mean |SHAP| share:", round(shap_info["reliability_share"], 6))
print("ranked:")
for name, imp in shap_info["ranked"]:
    print(f"  {name:24s} {imp:.6f}")

top4: ['is_proposed', 'count', 'match_mean', 'edge_spread']
reliability_mean |SHAP| share: 0.0
ranked:
  is_proposed              0.889862
  count                    0.573470
  match_mean               0.264480
  edge_spread              0.233822
  min_edge                 0.211839
  progress                 0.191616
  category_popularity      0.188912
  liquidity_min            0.136671
  liquidity_mean           0.131845
  size_spread              0.121676
  hours_since_created      0.111970
  hours_in_stage           0.092337
  vote_velocity            0.050229
  category_diversity       0.049309
  is_frozen                0.000000
  reliability_mean         0.000000


## 7. Oracle-анализ

Латентки из `raw_json` только для отчёта. Сравниваем SHAP с зашитыми эффектами DGP.
`fraud` и `|ε|` — irreducible error: `fraud` редок (~3%) и бьёт только `pBreak` на freeze;
`ε` — шум размера кластера, не прямой шум лейбла.

In [7]:
oracle = rt.oracle_analysis(split["df_test"], split["y_test"], p_test)
for k, v in oracle.items():
    print(f"{k}: {v}")

n: 4142
n_err: 793
fraud_rate_all: 0.027764365041042974
fraud_rate_err: 0.029003783102143757
err_rate_fraud: 0.2
err_rate_clean: 0.19120933697541595
abs_eps_all: 0.24110249840114129
abs_eps_err: 0.23621742914290833
err_rate_high_eps: 0.20173745173745175
err_rate_low_eps: 0.1880231809401159
n_fraud: 115
n_fraud_err: 23
reason_err: {'completed': 507, 'freeze_fail': 123, 'confirm_timeout': 98, 'proposed_timeout': 65}


## 8. Гейты и экспорт артефактов

Если любой гейт красный — `raise`, файлы модели / golden / отчёта не перезаписываются.

- `backend/pkg/utils/ranker/models/ranker_v1.txt` — LightGBM text (leaves)
- `ml/golden/golden_v1.json` — 20 пар {фичи, proba} с test
- `ml/reports/v1.md` — метрики, калибровка, SHAP, oracle

In [8]:
gates = rt.evaluate_gates(metrics, calib, shap_info)
for name, ok, detail in gates:
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

failed = [g for g in gates if not g[1]]
if failed:
    details = "; ".join(f"{n}: {d}" for n, _, d in failed)
    raise RuntimeError(f"гейты не пройдены, артефакты не записаны: {details}")

paths = rt.export_artifacts(
    ROOT, model, split, p_test, metrics, calib, shap_info, oracle, gates
)
print("exported:")
for k, p in paths.items():
    print(f"  {k}: {p}")

[PASS] delta_overall: AUC(LGBM)-AUC(formula)=0.1300 (want ≥ 0.02)
[PASS] delta_ADD: ADD delta=0.1941 (want ≥ 0.02)
[PASS] auc_cap: AUC(LGBM)=0.8801 (want < 0.95)
[PASS] calibration: max |pred-emp| on supported bins=0.0489 (want ≤ 0.05)
[PASS] shap_dgp: SHAP top4=['is_proposed', 'count', 'match_mean', 'edge_spread']; DGP overlap=['count', 'is_proposed', 'match_mean']
[PASS] reliability_shap: reliability_mean |SHAP| share=0.0000 (want ≈ 0)
exported:
  model: /home/dan/projects/tricky-exchanger/backend/pkg/ranker/models/ranker_v1.txt
  golden: /home/dan/projects/tricky-exchanger/ml/golden/golden_v1.json
  report: /home/dan/projects/tricky-exchanger/ml/reports/v1.md
  figures: /home/dan/projects/tricky-exchanger/ml/reports/figures
